In [ ]:
# Проект: Базовая ИИ для работы с текстом

In [ ]:
# Выполнили: Боровитинов Алексей, Попов Ярослав, Остроушко Илья

In [ ]:
# Цель проекта: обработка текстовых документов по инструкции пользователя
# Используемые технологии: Python, transformers, python-docx, модель Qwen
# Как пользоваться: загрузить модель, выбрать файл, ввести задачу, получить обработанный документ
# Ограничения: качество зависит от модели; длинные документы обрезаются; результат нужно проверять

In [2]:
import os
import warnings
# Глушим системные варнинги, чтобы консоль была чистой
warnings.filterwarnings('ignore')
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "true"

from transformers import pipeline

# Обычный рабочий принт для контроля запуска
print("Запуск инициализации NLP-модели...")

try:
    # Инициализируем легковесный пайплайн Qwen
    nlp_model = pipeline(
        "text-generation", 
        model="Qwen/Qwen2.5-0.5B-Instruct"
    )
    print("Статус: Модель успешно загружена.")
except Exception as err:
    print(f"Ошибка инициализации: {err}")

Запуск инициализации NLP-модели...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Статус: Модель успешно загружена.


In [ ]:
import os
from docx import Document

def main():
    print("============================================================")
    print("--- Панель обработки текстовых документов v1.4 ---")
    print("============================================================")
    # ВАЖНОЕ НАПОМИНАНИЕ О ДИРЕКТОРИИ
    print("⚠️  ВНИМАНИЕ: Файл ДОЛЖЕН лежать строго в папке с этим проектом!")
    print("============================================================")
    
    fname = input("Укажите имя файла (с расширением .txt или .docx): ").strip()
    print("-" * 50)
    
    if not os.path.exists(fname):
        print(f"Ошибка: Файл '{fname}' не найден в директории проекта.")
        print("Перенесите файл в папку к блокноту и повторите запуск.")
        return
        
    doc_text = ""
    
    # Чтение файлов .txt / .docx
    if fname.endswith('.txt'):
        try:
            with open(fname, 'r', encoding='utf-8') as f:
                doc_text = f.read()
        except UnicodeDecodeError:
            with open(fname, 'r', encoding='windows-1251') as f:
                doc_text = f.read()
                
    elif fname.endswith('.docx'):
        try:
            doc = Document(fname)
            doc_text = '\n'.join([p.text for p in doc.paragraphs])
        except Exception as e:
            print(f"Ошибка парсинга Word: {e}")
            return
    else:
        print("Формат файла не поддерживается.")
        return

    if doc_text.strip():
        print(f"Файл считан успешно. Объем данных: {len(doc_text)} симв.")
        task = input("Введите техническое задание (промт) для обработки: ").strip()
        print("-" * 50)
        
        # Берем большой кусок текста (до 15 000 слов), убрав жесткое ограничение
        words = doc_text.split()[:15000] 
        truncated_text = " ".join(words)
        
        # Оформляем строгий чат-контекст
        prompt_data = [
            {
                "role": "system", 
                "content": "Ты профессиональный текстовый редактор. Твоя единственная задача — выдать результат выполнения команды пользователя. Выводи ТОЛЬКО результат выполнения команды на русском языке. Не повторяй исходный текст."
            },
            {
                "role": "user",
                "content": f"ЗАДАНИЕ: {task}\n\nТЕКСТ ДЛЯ ОБРАБОТКИ:\n{truncated_text}"
            }
        ]
        
        print("Запущен процесс генерации ответа моделью...")
        print("=" * 50)
        
        try:
            # Генерация ответа с увеличенным лимитом до 1000 токенов
            raw_out = nlp_model(
                prompt_data, 
                max_new_tokens=1000, 
                do_sample=True, 
                temperature=0.4, 
                top_p=0.9
            )
            
            # ФИКС БАГА: Жесткое и безопасное извлечение ТОЛЬКО ответа ИИ
            # Новые модели возвращают структуру, где по ключу лежит сгенерированный текст
            if isinstance(raw_out, list) and len(raw_out) > 0:
                # Проверяем, вернулся ли нам сгенерированный диалог обратно
                generated_sequence = raw_out[0]
                if "generated_text" in generated_sequence:
                    chat_history = generated_sequence["generated_text"]
                    # Если внутри лежит список сообщений, забираем контент САМОГО ПОСЛЕДНЕГО (ответа ИИ)
                    if isinstance(chat_history, list):
                        out_text = chat_history[-1]["content"]
                    else:
                        # Если вернулась строка, то срезаем системную часть программно
                        out_text = str(chat_history)
                        if "ЗАДАНИЕ:" in out_text:
                            out_text = out_text.split("ТЕКСТ ДЛЯ ОБРАБОТКИ:")[-1]
                            # Отрезаем кусок исходного текста, оставляя только финал
                            if len(out_text.split('\n\n')) > 1:
                                out_text = "\n".join(out_text.split('\n\n')[1:])
                else:
                    out_text = str(raw_out)
            else:
                out_text = str(raw_out)
                
            print("🤖 РЕЗУЛЬТАТ ВЫПОЛНЕНИЯ ИИ:")
            print("-" * 50)
            print(out_text.strip())
            print("-" * 50)
            
            # Формируем имя нового файла с меткой _processed
            base_name, ext = os.path.splitext(fname)
            res_filename = f"{base_name}_processed{ext}"
            
            # Сохранение в зависимости от формата
            if fname.endswith('.txt'):
                with open(res_filename, 'w', encoding='utf-8') as f:
                    f.write(out_text.strip())
                print(f"Лог: Результат записан в файл: {res_filename}")
                
            elif fname.endswith('.docx'):
                new_doc = Document()
                for paragraph in out_text.strip().split('\n'):
                    if paragraph.strip():
                        new_doc.add_paragraph(paragraph)
                new_doc.save(res_filename)
                print(f"Лог: Сформирован новый документ Word: {res_filename}")
                
        except Exception as e:
            print(f"Сбой в процессе работы скрипта: {e}")
        print("=" * 50)
    else:
        print("Внимание: Выбранный файл не содержит текста.")

if __name__ == "__main__":
    main()


--- Панель обработки текстовых документов v1.4 ---
⚠️  ВНИМАНИЕ: Файл ДОЛЖЕН лежать строго в папке с этим проектом!
